# 00 — Why Normal Language Models Fail (and How RAG Fixes It)

**By the end of this notebook, you can explain why standalone LLMs hallucinate on private or recent facts and prove how supplying retrieved context fixes it.**

### Learning Objectives
- Understand four root limitations of standalone LLMs (knowledge cutoff, private data blindness, context limits, fine-tuning cost).
- Distinguish parametric memory (closed-book exam) from Retrieval-Augmented Generation (open-book exam).
- Observe an LLM failing without context versus answering accurately with retrieved context.

### Prerequisites
- None — designed for complete beginners who know basic Python.

## 1. Setup
Let's connect this notebook to our shared project helpers (`utils/models.py`), load `.env` (defaulting to free offline `local` mode), and initialize our LLM and embeddings.

In [1]:
# !pip install -q -r ../requirements.txt  # Uncomment if running in a fresh remote environment
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'utils').exists() else Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from utils.models import get_llm, get_embeddings, resolve_project_root
project_root = resolve_project_root()
data_dir = project_root / 'data'
llm = get_llm()
embeddings = get_embeddings()
print(f'Ready! Root={project_root.name} | LLM={type(llm).__name__}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready! Root=rag-from-scratch-course | LLM=LocalGroundedChatModel


## 2. Concept: Closed-Book vs. Open-Book AI

A **Large Language Model (LLM)** stores facts inside frozen model weights learned during training — like a student taking a **closed-book exam** using only what they memorized months ago.

When asked about *private company documents* or *events after its training cutoff*, a closed-book model suffers from **hallucination** (inventing plausible-sounding falsehoods rather than admitting ignorance). Furthermore, pasting every file into the prompt exceeds the model's **context window** (the maximum desk space of text it can read at once), while **fine-tuning** (re-training weights) is expensive and sluggish for fast-changing facts.

**Retrieval-Augmented Generation (RAG)** turns the AI into an **open-book exam**: first search your trusted documents for the exact 2-3 relevant snippets, then ask the LLM to answer *only* from those snippets.

```
[Question] --> (Closed-Book LLM) ------------------> Guesses / "No access to private data"

[Question] --> [Search Your Docs] --> (Top Snippets + Question) --> [Open-Book LLM] --> Grounded Answer!
```

## 3. Step-by-Step Build

### Step 1: Ask the LLM a Private Question WITHOUT Context
Let's ask our model about our fictional company Novasphere Dynamics' Enterprise Plan pricing without giving it any documents.

In [2]:
question = "How much does Novasphere Dynamics charge per month for the Enterprise Plan?"
closed_book = llm.invoke(question)
print("Closed-book reply:", closed_book.content)
        

Closed-book reply: I cannot verify internal or proprietary 2026 details about Novasphere Dynamics because I do not have access to private company documents. Supplying retrieved documents via RAG is required to give a grounded, factual answer.


> **What just happened?**
> Without retrieved documents, the model refuses to invent private pricing data (or in real cloud models, might hallucinate an outdated number).

### Step 2: Inspect the Actual Company Document
Now let's read the genuine internal file `data/company_handbook.txt` containing our company's private pricing rules.

In [3]:
handbook_text = (data_dir / "company_handbook.txt").read_text(encoding="utf-8")
for line in handbook_text.splitlines():
    if "Enterprise Plan" in line:
        print("Found in handbook:", line.strip())
        

Found in handbook: - Enterprise Plan: $4,200 per month. Supports up to 200 active nodes. Includes ZeroGravity automatic failover, QuantumCache v4, 1-year telemetry retention, and 24/7/365 mission-critical support.


> **What just happened?**
> The ground truth ($4,200/month) lives safely in our local text file—outside the LLM's frozen training weights.

### Step 3: Compare Context Window Limits vs. Relevant Snippets
Instead of dumping an entire 10,000-page handbook into the LLM's limited context window, let's check character counts and isolate only the pricing section.

In [4]:
pricing_snippet = "\n".join(
    line for line in handbook_text.splitlines() if "Plan:" in line or "PRICING" in line
)
print(f"Full handbook chars: {len(handbook_text)} | Focused snippet chars: {len(pricing_snippet)}")
        

Full handbook chars: 2908 | Focused snippet chars: 551


> **What just happened?**
> Focusing strictly on relevant paragraphs keeps prompts small, cheap, fast, and much more accurate.

### Step 4: Open-Book Answering with Grounded Context (Mini-RAG)
Now let's inject our focused snippet into `<context>` tags and instruct the LLM to answer solely from that evidence.

In [5]:
prompt = f"<context>\n{pricing_snippet}\n</context>\nQuestion: {question}"
open_book = llm.invoke(prompt)
print("Open-book reply:", open_book.content)
        

Open-book reply: Based on the provided context: Enterprise Plan: $4,200 per month. Supports up to 200 active nodes. Includes ZeroGravity automatic failover, QuantumCache v4, 1-year telemetry retention, and 24/7/365 mission-critical support. 4. COMMERCIAL PRICING PLANS (NOVASPHERE OS) Starter Plan: $499 per month. Supports up to 10 active robot or drone nodes. Includes standard telemetry dashboards and 30-day log retention. Professional Plan: $1,850 per month. Supports up to 50 active nodes. Includes the HyperSync protocol, 90-day telemetry retention, and dedicated Slack Connect support.


> **What just happened?**
> Once grounded in retrieved text, the LLM returns the exact figure ($4,200 per month) with zero guessing!

## 4. Try It Yourself
Experiment by editing the starter ideas below before peeking at the solution!

1. **Test a recent 2026 event** — *Hint:* Read `recent_changelog.md` and ask who was appointed Chief AI Ethics Officer with vs. without context.
2. **Try an unanswerable question** — *Hint:* Pass the pricing snippet and ask for Novasphere's stock ticker symbol on NASDAQ.

<details>
<summary><strong>Click to reveal sample solution</strong></summary>

Run the cell below to verify your solution.
</details>

In [6]:
changelog = (data_dir / "recent_changelog.md").read_text(encoding="utf-8")
print("No context :", llm.invoke("Who is Novasphere's Chief AI Ethics Officer?").content[:95])
print("With RAG   :", llm.invoke(f"<context>\n{changelog}\n</context>\nWho was appointed Chief AI Ethics Officer?").content)
print("Missing    :", llm.invoke(f"<context>\n{changelog}\n</context>\nWhat is Novasphere's NASDAQ ticker?").content)
        

No context : I cannot verify internal or proprietary 2026 details about Novasphere Dynamics because I do not
With RAG   : Based on the provided context: ### Leadership Announcement (October 2026) Novasphere Dynamics is pleased to announce that **Dr. Elina Vance** has been appointed **Chief AI Ethics Officer**, overseeing autonomous safety auditing, algorithmic transparency, and EU AI Act compliance. ### Edge Memory Optimization: QuantumCache v4 Reduced resident RAM usage on NVIDIA Jetson Orin edge controllers by 30% during peak Lidar burst capture. ### Leadership Announcement (October 2026) Novasphere Dynamics is pleased to announce that **Dr. Elina Vance** has been appointed **Chief AI Ethics Officer**, overseeing autonomous safety auditing, algorithmic transparency, and EU AI Act compliance.
Missing    : I don't know based on the provided context. The supplied documents do not contain enough information to answer this question.


## 5. Common Mistakes
- Assuming fine-tuning updates a model's factual memory cheaply — fine-tuning shapes tone and task style, while RAG supplies accurate, updatable facts.
- Stuffing entire 100-page PDFs into one prompt — causes lost-in-the-middle errors and huge token bills.
- Prompting without telling the model to say 'I don't know' when context lacks the answer.

## 6. Recap
- Standalone LLMs suffer from training cutoffs, private-data blindness, hallucinations, and context-window budgets.
- RAG pairs an external search step with a grounded generation step (open-book exam).
- You saw firsthand how supplying `<context>` turns an unknown proprietary question into a verified fact.

## 7. Next Notebook
Next up (`01_hello_rag.ipynb`): Build an end-to-end automated RAG pipeline using real LangChain loaders, splitters, embeddings, and vector search!

## 8. Video Chapter Outline
1. **The Closed-Book Problem** — Why LLMs forget, hallucinate, and hit training cutoffs.
2. **Context Window & Fine-Tuning Myths** — Why dumping huge files or retraining fails for factual recall.
3. **Open-Book AI Intuition** — Walkthrough of Retrieval-Augmented Generation with a library analogy.
4. **Live Code Demo** — Closed-book failure vs. open-book grounded success on Novasphere data.
5. **Wrap-up & What's Next** — Key takeaways before building automated search in NB01.